# 1. Introduction

The following notebook contains the data preparation for the subsequent analysis. 
The data consists of two datasets: information on electric vehicle charging sessions and the corresponding weather data.
Before the data can be used, it must be checked and, if necessary, adjusted. The goal is to obtain a cleaned, consistent dataset so that we have a suitable basis for further analysis. This is achieved through the following steps:

   - Data Assessment
   - Duplicates
   - Missing Values
   - Data Types
   - Data Cleaning & Filtering
   - Feature Engineering
   - Restructuring Data
   - Final Data Quality Check
   - Train-Test Split


# 2. Import Libraries & Load Data 
## 2.1 Import Libraries

In [182]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2.2 Load Data

In [183]:
sessions = pd.read_csv("../data/charging_sessions.csv")
weather = pd.read_csv("../data/weather_burbank_airport.csv")
display(sessions.head())
display(weather.head())

,Unnamed: 0,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,0,5e23b149f9af8b5fe4b973cf,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,194.0,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil..."
1,1,5e23b149f9af8b5fe4b973d0,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,1,AG-1F01,1-1-193-825,America/Los_Angeles,4275.0,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil..."
2,2,5e23b149f9af8b5fe4b973d1,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,1,AG-1F03,1-1-193-829,America/Los_Angeles,344.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
3,3,5e23b149f9af8b5fe4b973d2,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,1,AG-1F04,1-1-193-820,America/Los_Angeles,1117.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
4,4,5e23b149f9af8b5fe4b973d3,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,1,AG-1F06,1-1-193-819,America/Los_Angeles,334.0,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil..."


,city,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
0,Burbank,2018-01-01 08:53:00,9.0,33.0,Fair,991.75,9.0,0.0,8.0
1,Burbank,2018-01-01 09:53:00,9.0,33.0,Fair,992.08,0.0,0.0,9.0
2,Burbank,2018-01-01 10:53:00,9.0,21.0,Haze,992.08,0.0,0.0,9.0
3,Burbank,2018-01-01 11:53:00,9.0,29.0,Partly Cloudy,992.08,0.0,0.0,9.0
4,Burbank,2018-01-01 12:53:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0


# 3. Data Preparation

# Data Preparation

* Charging Sessions

  @Description of Fields of Charging Session Dataset
  
  *  **basic infos**  
  timezone  ——
  siteID    ——   Unique identifier for the site.   
  spaceID   ——     Unique identifier of the parking space.   
  stationID  ——   Unique identifier of the charging point.   

  *  **user level**  
  userID    ——   Unique identifier of the user, if provided.   
  userInputs  —— only available if the user participates in app-based charging.  
   
   *  **session level** 
   *  **recorded session**  /string   
   id —— Unique identifier of the session record   
   sessionID —— Unique identifier for the session.   
   
    *  **charging duration**  /datetime  
      * requested charging duration    
        modifiedAt  —— Time this user input was provided.   
        requestedDeparture —— User estimated departure time. 
        minutesAvailable —— Length of the session as estimated by the user.   
      * recorded charging duration   
        connectionTime —— Time when the EV plugged in.   
        disconnectTime —— Time when the EV unplugged.   
        doneChargingTime —— Time of the last non-zero current draw recorded.   
    
    *  **charging energy and driving range**  /float  
      * requested charging energy and driving range  
        kWhRequested —— Energy requested by the user in kWh.   
        milesRequested —— Number of miles requested by the user.     
      * recorded charging energy  
        kWhDelivered —— Amount of energy delivered during the session.  
      * Efficiency  
        WhPerMile —— Efficiency of the EV in Wh per mile.   

    *  **payment**  /bool  
     paymentRequired —— If the user was required to pay for this session.   

    
## 3.1 Data Assessment


In [184]:
# Overview of charging sessions
print("Charging Sessions:")
print(sessions.info())
print("Shape:", sessions.shape)

# Overview of weather data
print("\nWeather Data:")
print(weather.info())
print("Shape:", weather.shape)

Charging Sessions:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66450 entries, 0 to 66449
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        66450 non-null  int64  
 1   id                66450 non-null  object 
 2   connectionTime    66450 non-null  object 
 3   disconnectTime    66450 non-null  object 
 4   doneChargingTime  62362 non-null  object 
 5   kWhDelivered      66450 non-null  float64
 6   sessionID         66450 non-null  object 
 7   siteID            66450 non-null  int64  
 8   spaceID           66450 non-null  object 
 9   stationID         66450 non-null  object 
 10  timezone          66450 non-null  object 
 11  userID            49187 non-null  float64
 12  userInputs        49187 non-null  object 
dtypes: float64(2), int64(2), object(9)
memory usage: 6.6+ MB
None
Shape: (66450, 13)

Weather Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29244 entries, 0

## 3.2 Duplicates

First, the session IDs are checked for duplicate rows, and then the system checks to see if there are any duplicate session IDs. This process is then repeated with the weather data.

In [185]:
print("Charging Sessions:")
print("Duplicate rows:", sessions.duplicated().sum())
print("Duplicate Session-IDs:", sessions.duplicated(subset=["sessionID"]).sum())
dups = sessions[sessions.duplicated(subset=["sessionID"], keep=False)].sort_values("sessionID")
print("Affected Rows:", len(dups))
print("Affected IDs:", dups["sessionID"].nunique())

print("\nWeather Data:")
print("Duplicate rows:", weather.duplicated().sum())
print("Duplicate Weather Timestamps:", weather.duplicated(subset=["timestamp"]).sum())

Charging Sessions:
Duplicate rows: 0
Duplicate Session-IDs: 1413
Affected Rows: 2826
Affected IDs: 1413

Weather Data:
Duplicate rows: 0
Duplicate Weather Timestamps: 0


No duplicates were found in the weather data. However, entries with the same ID were found in “Charging sessions.” The next step is to check whether these are export errors. For this purpose, the rows are compared to determine whether they are identical except for the Unnamed: 0 column ( index saved during the CSV export).


In [186]:
columns_to_compare = sessions.columns.drop("Unnamed: 0")

print("Duplicates without Unnamed: 0:",
      sessions.duplicated(subset=columns_to_compare).sum())

Duplicates without Unnamed: 0: 1413


These lines are, in fact, export errors, so they will be deleted.

In [187]:
sessions = sessions.drop(columns=["Unnamed: 0"])
sessions = sessions.drop_duplicates()
print("Duplicate rows:", sessions.duplicated().sum())

Duplicate rows: 0


## 3.3 Data Cleaning & Filtering

Drop Unnecessary columns and rows

For the charging df:
- Drop timezone as it is the same value for all rows
- Drop stationID and sessionID as they contain no new information (e.g. if siteID is one, the prefix of the stationID starts with 'AG' and for 2 it starts with 'CA')
- "id" column also does not contain information relevant for Data Analytics

In [188]:
# use only spaceID since siteID, spaceID and stationID are dependent
sessions = sessions.drop(columns=['stationID'])
sessions = sessions.drop(columns=['sessionID'])

# is the a entry where timezone is not los angeles?
sessions.groupby('timezone').count()
# since the value is always los angeles and there are no missing values, we can drop the column
sessions = sessions.drop(columns=['timezone'])
# drop id column
sessions = sessions.drop(columns=['id'])

sessions.head()

,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,siteID,spaceID,userID,userInputs
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1,AG-3F30,194.0,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil..."
1,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1,AG-1F01,4275.0,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil..."
2,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1,AG-1F03,344.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
3,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1,AG-1F04,1117.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
4,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1,AG-1F06,334.0,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil..."


For the weather df:
- drop city column as the value is always burbank

In [189]:
weather[weather["city"] != "Burbank"].count()

weather = weather.drop(columns=['city'])

## 3.4 Missing values
In the next step the datasets are checked for missing values.

In [190]:
# Charging sessions
print("Charging Sessions:")
display(sessions.isnull().sum().sort_values(ascending=False))

# Weather data
print("Weather Data:")
display(weather.isnull().sum().sort_values(ascending=False))


Charging Sessions:


userID              17215
userInputs          17215
doneChargingTime     4087
connectionTime          0
kWhDelivered            0
disconnectTime          0
spaceID                 0
siteID                  0
dtype: int64

Weather Data:


windspeed                  86
felt_temperature           26
temperature                25
cloud_cover                20
cloud_cover_description    20
pressure                    8
timestamp                   0
precipitation               0
dtype: int64

It is notable that `userID` and `userInputs` are missing with equal frequency. Therefore, it is checked whether this occurs in the same rows.
If this is the case, it could be due to unregistered users.

In [191]:
print("Is it the same rows:",
      (sessions["userID"].isnull() == sessions["userInputs"].isnull()).all())


Is it the same rows: True


Since these are the same columns, it is assumed that they are unregistered users. A flag  is introduced for this purpose.


In [192]:
sessions["is_registered"] = sessions["userID"].notnull()
print(sessions["is_registered"].value_counts())


is_registered
True     47822
False    17215
Name: count, dtype: int64


In the next step, `doneChargingTime` will be examined.

In [193]:
print(sessions.groupby(sessions["doneChargingTime"].isnull())["kWhDelivered"].describe())

                    count       mean        std    min     25%     50%  \
doneChargingTime                                                         
False             60950.0  11.583268  10.153601  0.501  5.0130   8.991   
True               4087.0  13.986472  11.767162  0.502  5.7845  10.899   

                     75%         max  
doneChargingTime                      
False             13.984  108.797242  
True              18.145   77.700000  


In [194]:
sessions["doneChargingTime"] = sessions["doneChargingTime"].fillna(sessions["disconnectTime"])

still_missing = sessions[sessions["doneChargingTime"].isna()]
print(f"{len(still_missing)} Zeilen ohne doneChargingTime")
still_missing

0 Zeilen ohne doneChargingTime


,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,siteID,spaceID,userID,userInputs,is_registered


In [195]:
print(sessions.groupby(sessions["doneChargingTime"] == sessions["disconnectTime"])["kWhDelivered"].describe())

         count       mean        std    min      25%     50%      75%  \
False  60945.0  11.583387  10.153710  0.501  5.01300   8.991  13.9840   
True    4092.0  13.981767  11.764582  0.502  5.78775  10.885  18.1435   

              max  
False  108.797242  
True    77.700000  


In [196]:
dur_h = (pd.to_datetime(sessions["doneChargingTime"]) - pd.to_datetime(sessions["connectionTime"])).dt.total_seconds() / 3600

print(dur_h[dur_h > 0].groupby(sessions["doneChargingTime"] == sessions["disconnectTime"]).describe())

         count      mean       std       min       25%       50%       75%  \
False  60916.0  3.526389  3.064319  0.000833  1.619722  2.760833  4.726250   
True    4092.0  4.742509  4.086069  0.034444  1.029236  3.878889  8.393889   

              max  
False  200.015833  
True    57.457222  


### TODO: Shoudl we instead use an average funktion to replace the values? Using the disconnectTime is about an hour longer than the mean. This would of course reduce variance but it might increase usefullness for analysis such as the clustering

The `doneChargingTime` value is missing for approximately 6.3% of the sessions. A comparison of the energy charged shows that these sessions do not differ significantly from the rest (median 10.9 vs. 9.0 kWh, minimum approximately 0.5 kWh in each case). These are therefore regular charging processes for which the end of charging was not recorded. The values are left blank so as not to skew the charging duration. It will be calculated later only for sessions with available values.

In addition to the columns already examined, the weather data also contain missing values, though only a few (at most 86 per column; see above). Since the weather data constitute a time series, the numerical columns are sorted by timestamp and linearly interpolated. The cloud cover value is filled in with the last known value.

In [197]:
weather["timestamp"] = pd.to_datetime(weather["timestamp"])
weather = weather.sort_values("timestamp").reset_index(drop=True)

num_cols = ["windspeed", "felt_temperature", "temperature", "cloud_cover", "pressure"]
weather[num_cols] = weather[num_cols].interpolate(method="linear")
weather["cloud_cover_description"] = weather["cloud_cover_description"].ffill()

print(weather.isnull().sum())

timestamp                  0
temperature                0
cloud_cover                0
cloud_cover_description    0
pressure                   0
windspeed                  0
precipitation              0
felt_temperature           0
dtype: int64


## 3.5 Transform UserInputs

In [199]:
sessions.head()

,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,siteID,spaceID,userID,userInputs,is_registered
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1,AG-3F30,194.0,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil...",True
1,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1,AG-1F01,4275.0,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil...",True
2,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1,AG-1F03,344.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile...",True
3,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1,AG-1F04,1117.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile...",True
4,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1,AG-1F06,334.0,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil...",True


If there are multiple UserInputs, the entry with the latest "modifedAt" value is taken as we thought this would most likely be the most correct one out of all.

In [200]:
import ast

# userInput structure with example data:
# [{'WhPerMile': 250, 'kWhRequested': 25.0, 'milesRequested': 100, 'minutesAvailable': 463, 'modifiedAt': 'Thu, 02 Jan 2020 13:09:39 GMT', 'paymentRequired': True, 'requestedDeparture': 'Thu, 02 Jan 2020 20:51:54 GMT', 'userID': 194}]

user_inputs = sessions["userInputs"].fillna("[{'WhPerMile': -1, 'kWhRequested': -1, 'milesRequested': -1, 'minutesAvailable': -1, 'modifiedAt': '', 'paymentRequired':'' , 'requestedDeparture': '', 'userID': 0}]")

whPerMile = []
kwhRequested = []
milesRequested = []
minutesAvailable = []
modifiedAt = []

for row in user_inputs:
    dicts = []
    if pd.notna(row):
        parsed = ast.literal_eval(row)
        if parsed:
            latest = max(parsed, key=lambda d: d["modifiedAt"]) 
            dicts.append(latest)

        for d in dicts:
            for key, value in d.items():
                whPerMile.append(value) if key == 'WhPerMile' else None
                kwhRequested.append(value) if key == 'kWhRequested' else None
                milesRequested.append(value) if key == 'milesRequested' else None
                minutesAvailable.append(value) if key == 'minutesAvailable' else None
                modifiedAt.append(value) if key == 'modifiedAt' else None

sessions['WhPerMile'] = whPerMile
sessions['kWhRequested'] = kwhRequested
sessions['milesRequested'] = milesRequested
sessions['minutesAvailable'] = minutesAvailable
sessions['modifiedAt'] = modifiedAt

# drop userInputs as the values have been transformed into own columns
sessions = sessions.drop(columns=['userInputs'])

In [201]:
sessions.head(20)

,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,siteID,spaceID,userID,is_registered,WhPerMile,kWhRequested,milesRequested,minutesAvailable,modifiedAt
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1,AG-3F30,194.0,True,250,25.00,100,463,"Thu, 02 Jan 2020 13:09:39 GMT"
1,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1,AG-1F01,4275.0,True,280,70.00,250,595,"Thu, 02 Jan 2020 13:37:11 GMT"
2,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1,AG-1F03,344.0,True,400,8.00,20,60,"Thu, 02 Jan 2020 13:57:17 GMT"
3,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1,AG-1F04,1117.0,True,400,8.00,20,65,"Thu, 02 Jan 2020 14:00:19 GMT"
4,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1,AG-1F06,334.0,True,400,16.00,40,504,"Thu, 02 Jan 2020 14:00:13 GMT"
5,2020-01-02 14:00:13+00:00,2020-01-02 23:00:41+00:00,2020-01-02 20:13:21+00:00,13.370,1,AG-1F13,3519.0,True,600,24.00,40,624,"Thu, 02 Jan 2020 14:00:12 GMT"
6,2020-01-02 14:09:14+00:00,2020-01-03 01:57:58+00:00,2020-01-03 00:59:21+00:00,43.477,1,AG-1F09,933.0,True,385,65.45,170,647,"Thu, 02 Jan 2020 14:09:33 GMT"
7,2020-01-02 14:17:32+00:00,2020-01-03 00:44:54+00:00,2020-01-02 20:22:40+00:00,32.441,1,AG-4F34,3737.0,True,300,45.00,150,126,"Thu, 02 Jan 2020 14:17:40 GMT"
8,2020-01-02 14:25:38+00:00,2020-01-02 19:36:05+00:00,2020-01-02 18:38:39+00:00,13.264,1,AG-1F11,419.0,True,400,40.00,100,491,"Thu, 02 Jan 2020 14:25:45 GMT"
9,2020-01-02 14:27:40+00:00,2020-01-02 20:01:48+00:00,2020-01-02 19:17:46+00:00,13.285,1,AG-1F08,651.0,True,400,20.00,50,226,"Thu, 02 Jan 2020 14:27:48 GMT"


In [202]:
sessions[sessions["WhPerMile"] == -1].count()
# this is how to filter the originally null values

connectionTime      17215
disconnectTime      17215
doneChargingTime    17215
kWhDelivered        17215
siteID              17215
spaceID             17215
userID                  0
is_registered       17215
WhPerMile           17215
kWhRequested        17215
milesRequested      17215
minutesAvailable    17215
modifiedAt          17215
dtype: int64

Now fill up the numercial null values by taking averages

In [203]:
for col in ["WhPerMile", "kWhRequested", "milesRequested", "minutesAvailable"]:
    sessions[col] = sessions[col].astype(float)   # mean is a float, int columns cannot hold it
    mean_val = sessions.loc[sessions[col] != -1, col].mean()
    sessions.loc[sessions[col] == -1, col] = mean_val

sessions[sessions["WhPerMile"] == -1].count()
sessions.head(20)

C:\Users\malen\AppData\Local\Temp\ipykernel_21692\2374755378.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '361.1973986867969' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sessions.loc[sessions[col] == -1, col] = mean_val
C:\Users\malen\AppData\Local\Temp\ipykernel_21692\2374755378.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '74.95773911588809' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sessions.loc[sessions[col] == -1, col] = mean_val
C:\Users\malen\AppData\Local\Temp\ipykernel_21692\2374755378.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '357.0206808581824' has dtype incompatible with int64, please explicitly cast to a compatible dty

,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,siteID,spaceID,userID,is_registered,WhPerMile,kWhRequested,milesRequested,minutesAvailable,modifiedAt
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1,AG-3F30,194.0,True,250.000000,25.00000,100.000000,463.000000,"Thu, 02 Jan 2020 13:09:39 GMT"
1,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1,AG-1F01,4275.0,True,280.000000,70.00000,250.000000,595.000000,"Thu, 02 Jan 2020 13:37:11 GMT"
2,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1,AG-1F03,344.0,True,400.000000,8.00000,20.000000,60.000000,"Thu, 02 Jan 2020 13:57:17 GMT"
3,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1,AG-1F04,1117.0,True,400.000000,8.00000,20.000000,65.000000,"Thu, 02 Jan 2020 14:00:19 GMT"
4,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1,AG-1F06,334.0,True,400.000000,16.00000,40.000000,504.000000,"Thu, 02 Jan 2020 14:00:13 GMT"
5,2020-01-02 14:00:13+00:00,2020-01-02 23:00:41+00:00,2020-01-02 20:13:21+00:00,13.370,1,AG-1F13,3519.0,True,600.000000,24.00000,40.000000,624.000000,"Thu, 02 Jan 2020 14:00:12 GMT"
6,2020-01-02 14:09:14+00:00,2020-01-03 01:57:58+00:00,2020-01-03 00:59:21+00:00,43.477,1,AG-1F09,933.0,True,385.000000,65.45000,170.000000,647.000000,"Thu, 02 Jan 2020 14:09:33 GMT"
7,2020-01-02 14:17:32+00:00,2020-01-03 00:44:54+00:00,2020-01-02 20:22:40+00:00,32.441,1,AG-4F34,3737.0,True,300.000000,45.00000,150.000000,126.000000,"Thu, 02 Jan 2020 14:17:40 GMT"
8,2020-01-02 14:25:38+00:00,2020-01-02 19:36:05+00:00,2020-01-02 18:38:39+00:00,13.264,1,AG-1F11,419.0,True,400.000000,40.00000,100.000000,491.000000,"Thu, 02 Jan 2020 14:25:45 GMT"
9,2020-01-02 14:27:40+00:00,2020-01-02 20:01:48+00:00,2020-01-02 19:17:46+00:00,13.285,1,AG-1F08,651.0,True,400.000000,20.00000,50.000000,226.000000,"Thu, 02 Jan 2020 14:27:48 GMT"


## 3.6 Data Types
In this step, the data types are reviewed and checked for validity. They may be adjusted if necessary.

In [204]:
print("Charging Sessions - Data Types:")
print(sessions.dtypes)
print()
print("Weather Data - Data Types:")
print(weather.dtypes)

Charging Sessions - Data Types:
connectionTime       object
disconnectTime       object
doneChargingTime     object
kWhDelivered        float64
siteID                int64
spaceID              object
userID              float64
is_registered          bool
WhPerMile           float64
kWhRequested        float64
milesRequested      float64
minutesAvailable    float64
modifiedAt           object
dtype: object

Weather Data - Data Types:
timestamp                  datetime64[ns]
temperature                       float64
cloud_cover                       float64
cloud_cover_description            object
pressure                          float64
windspeed                         float64
precipitation                     float64
felt_temperature                  float64
dtype: object


In [205]:
time_cols = ["connectionTime", "disconnectTime", "doneChargingTime"]
for col in time_cols:
    sessions[col] = pd.to_datetime(sessions[col], utc=True)


sessions["userID"] = sessions["userID"].astype("Int64")


print(sessions.dtypes)

connectionTime      datetime64[ns, UTC]
disconnectTime      datetime64[ns, UTC]
doneChargingTime    datetime64[ns, UTC]
kWhDelivered                    float64
siteID                            int64
spaceID                          object
userID                            Int64
is_registered                      bool
WhPerMile                       float64
kWhRequested                    float64
milesRequested                  float64
minutesAvailable                float64
modifiedAt                       object
dtype: object


The time columns of the charging sessions were stored as text and have been converted to datetime (UTC). userID was stored as a float because of the missing values and has been converted to a nullable integer. siteID is an identifier, not a numeric measure, and was converted to a categorical column. The weather timestamp was already converted during the interpolation step.

Convert "cloud_cover_description" to categorical columns

In [206]:
# convert cloud_cover_description to categorical
weather.groupby('cloud_cover_description').count()

cloud_cover_dummies = pd.get_dummies(weather['cloud_cover_description']).astype(int)

weather.join(cloud_cover_dummies).drop(columns=['cloud_cover_description']).head(20)

,timestamp,temperature,cloud_cover,pressure,windspeed,precipitation,felt_temperature,Blowing Dust,Cloudy,Cloudy / Windy,...,Mostly Cloudy,Mostly Cloudy / Windy,Partly Cloudy,Partly Cloudy / Windy,Rain,Rain / Windy,Smoke,T-Storm,Thunder,Thunder in the Vicinity
0,2018-01-01 08:53:00,9.0,33.0,991.75,9.0,0.0,8.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2018-01-01 09:53:00,9.0,33.0,992.08,0.0,0.0,9.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2018-01-01 10:53:00,9.0,21.0,992.08,0.0,0.0,9.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2018-01-01 11:53:00,9.0,29.0,992.08,0.0,0.0,9.0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,2018-01-01 12:53:00,8.0,33.0,992.08,0.0,0.0,8.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,2018-01-01 13:53:00,8.0,33.0,992.08,0.0,0.0,8.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,2018-01-01 14:53:00,7.0,30.0,992.08,0.0,0.0,7.0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
7,2018-01-01 15:53:00,8.0,34.0,992.41,0.0,0.0,8.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,2018-01-01 16:53:00,12.0,34.0,993.39,0.0,0.0,12.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,2018-01-01 17:53:00,16.0,34.0,994.05,0.0,0.0,16.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Remove Negative Charging Times

In [207]:
dur_h = (pd.to_datetime(sessions["doneChargingTime"]) - pd.to_datetime(sessions["connectionTime"])).dt.total_seconds() / 3600

neg = sessions[dur_h < 0].assign(dauer_min=dur_h[dur_h < 0] * 60)
print(f"{len(neg)} Sessions mit negativer Dauer")
neg[["spaceID", "connectionTime", "doneChargingTime", "disconnectTime", "kWhDelivered", "dauer_min"]].sort_values("dauer_min").head(20)

27 Sessions mit negativer Dauer


,spaceID,connectionTime,doneChargingTime,disconnectTime,kWhDelivered,dauer_min
44041,CA-307,2018-11-04 03:56:17+00:00,2018-11-04 03:14:55+00:00,2018-11-04 04:15:00+00:00,0.521000,-41.366667
44042,CA-303,2018-11-04 04:33:19+00:00,2018-11-04 03:52:29+00:00,2018-11-04 04:52:37+00:00,2.051000,-40.833333
44040,CA-495,2018-11-04 02:10:08+00:00,2018-11-04 01:40:31+00:00,2018-11-04 02:55:22+00:00,0.988647,-29.616667
27740,CA-322,2019-09-05 18:44:57+00:00,2019-09-05 18:43:57+00:00,2019-09-06 00:55:19+00:00,1.067230,-1.000000
41263,CA-494,2018-09-27 22:58:45+00:00,2018-09-27 22:57:45+00:00,2018-09-28 21:58:26+00:00,4.443405,-1.000000
42102,CA-493,2018-10-08 19:14:32+00:00,2018-10-08 19:13:32+00:00,2018-10-09 00:54:50+00:00,1.104249,-1.000000
42342,CA-322,2018-10-11 15:58:47+00:00,2018-10-11 15:57:47+00:00,2018-10-11 20:06:02+00:00,0.714337,-1.000000
31287,CA-494,2018-05-04 19:23:52+00:00,2018-05-04 19:22:52+00:00,2018-05-05 00:04:15+00:00,0.912297,-1.000000
42618,CA-322,2018-10-15 15:42:14+00:00,2018-10-15 15:41:14+00:00,2018-10-15 20:39:47+00:00,0.858423,-1.000000
43484,CA-321,2018-10-26 20:27:53+00:00,2018-10-26 20:26:53+00:00,2018-10-27 00:56:04+00:00,0.871274,-1.000000


The 27 sessions above have a `doneChargingTime` before the `connectionTime`. Charging cannot end before the car is plugged in, so these are recording errors and are removed.

In addition, 4,387 sessions have a `doneChargingTime` slightly after the `disconnectTime` (median about 1 minute, maximum about 60 minutes). This is most likely a small delay in logging the last current draw. Since there are too many to drop and the error is small, `doneChargingTime` is capped at `disconnectTime`. For these sessions, charging stopped when the car was unplugged.

In [ ]:
# TODO (discuss in group): proposal for handling invalid charging times, see the TODO below.
# - drop the 27 sessions with doneChargingTime < connectionTime (recording errors)
# - cap doneChargingTime at disconnectTime for the 4,387 sessions where it is slightly later (logging delay)
# Without this, charging_hours and idle_hours in 3.7 contain negative values.

# Drop sessions where charging ends before the car was plugged in
sessions = sessions[sessions["doneChargingTime"] >= sessions["connectionTime"]]

# Cap doneChargingTime at disconnectTime (charging cannot end after unplugging)
sessions["doneChargingTime"] = sessions[["doneChargingTime", "disconnectTime"]].min(axis=1)

print("Sessions left:", len(sessions))
print("Negative charging times:", (sessions["doneChargingTime"] < sessions["connectionTime"]).sum())
print("Charging ends after disconnect:", (sessions["doneChargingTime"] > sessions["disconnectTime"]).sum())

### Transform userInputs into own column

## TODO: I would like your opinions on if or how we should change these rows
I think we should probably drop these 27 rows.
Also, that we should drop the one row with the overlapping parking spot in which there are no user data. 

## 3.7 Feature Engineering
Add columns, that might be useful (e.g duration, holidays)

The following features are derived from the charging sessions:
- `session_hours`: total time the EV was plugged in (connection to disconnection)
- `charging_hours`: time until the last non-zero current draw (connection to doneChargingTime)
- `idle_hours`: time the EV stayed connected after charging had finished

Since the timestamps were converted to UTC in 3.6 and all stations are located in the `America/Los_Angeles` timezone (see 3.3), the connection time is converted to local time before the time features are extracted. Otherwise the arrival hour would be shifted by 7–8 hours and some sessions would end up on the wrong weekday.

In [ ]:
# Durations in hours
sessions["session_hours"]  = (sessions["disconnectTime"]   - sessions["connectionTime"]).dt.total_seconds() / 3600
sessions["charging_hours"] = (sessions["doneChargingTime"] - sessions["connectionTime"]).dt.total_seconds() / 3600
sessions["idle_hours"]     = sessions["session_hours"] - sessions["charging_hours"]

# Time features (local time)
# timestamps are stored in UTC, so convert to the local timezone of the stations first
local_connection = sessions["connectionTime"].dt.tz_convert("America/Los_Angeles")

sessions["arrival_hour"] = local_connection.dt.hour
sessions["weekday"]      = local_connection.dt.dayofweek   # 0 = Monday
sessions["is_weekend"]   = sessions["weekday"] >= 5
sessions["month"]        = local_connection.dt.month
sessions["year"]         = local_connection.dt.year

sessions[["session_hours", "charging_hours", "idle_hours", "arrival_hour", "weekday", "month", "year"]].describe()

### Holidays
On public holidays the charging stations are used much less, similar to a weekend. A holiday flag is therefore added, using the holiday dates for 2018–2021 (the period covered by the charging sessions).


Not every entry in the holiday file is a day off. Days on which usage stayed at a normal weekday level (Valentine's Day, Thanksgiving Eve, Columbus Day, Veterans Day, Juneteenth) are not counted. Easter and Labor Day Weekend always fall on a weekend and therefore make no difference. The following days are counted as holidays:
- New Year's Day, Martin Luther King Jr. Day, Washington's Birthday
- Memorial Day, 4th of July, Labor Day
- Thanksgiving Day, Christmas Eve, Christmas Day, New Year's Eve

Two days that are missing from the file are added:
- **Day after Thanksgiving:** not an official federal holiday, but usage at both sites drops by about 90% every year, to weekend level.
- **Observed 4th of July:** in 2020 the 4th of July fell on a Saturday, so the holiday was observed on Friday, 3 July. In 2021 it fell on a Sunday and was observed on Monday, 5 July.

`is_weekend` is kept unchanged. The new column `is_day_off` combines weekends and holidays.

Source: https://www.kaggle.com/datasets/donnetew/us-holiday-dates-2004-2021?resource=download

In [1]:

holidays = pd.read_csv("../data/US Holiday Dates (2018-2021).csv", sep=";", skiprows=1)
holidays["Date"] = pd.to_datetime(holidays["Date"])

# only days that are actually days off (note: the file uses a curly apostrophe in "New Year’s Eve")
days_off = ["New Year's Day", "Martin Luther King, Jr. Day", "Washington's Birthday",
            "Memorial Day", "4th of July", "Labor Day", "Thanksgiving Day",
            "Christmas Eve", "Christmas Day", "New Year’s Eve"]
missing = set(days_off) - set(holidays["Holiday"])
assert not missing, f"Holidays not found in file: {missing}"
holiday_dates = holidays.loc[holidays["Holiday"].isin(days_off), "Date"]

# add days missing in the file: day after Thanksgiving and the observed 4th of July
# (2020: Saturday -> Friday 3 July, 2021: Sunday -> Monday 5 July)
day_after_thanksgiving = holidays.loc[holidays["Holiday"] == "Thanksgiving Day", "Date"] + pd.Timedelta(days=1)
holiday_dates = pd.concat([holiday_dates, day_after_thanksgiving, pd.Series(pd.to_datetime(["2020-07-03", "2021-07-05"]))])

# compare on the local calendar date of the connection time
local_date = local_connection.dt.tz_localize(None).dt.normalize()
sessions["is_holiday"] = local_date.isin(holiday_dates)
sessions["is_day_off"] = sessions["is_weekend"] | sessions["is_holiday"]

print("Holiday dates:", holiday_dates.nunique())
print("Sessions on holidays:", sessions["is_holiday"].sum())
print("Sessions on holidays that fall on a weekday:", (sessions["is_holiday"] & ~sessions["is_weekend"]).sum())
print(sessions["is_day_off"].value_counts())

NameError: name 'pd' is not defined

## 3.9 Final Data Quality Check